In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
df= pd.read_csv("/content/drive/MyDrive/w2149468_5DATA004C_IndividualCW_Dataset/UNFCCC_v30_2016-2024.csv")

In [ ]:
df.head(7)

,Country_code,Country,Format_name,Pollutant_name,Sector_code,Sector_name,Parent_sector_code,Unit,Year,emissions,Notation,PublicationDate,DataSource,Country_code_3
0,BG,Bulgaria,IPCC Common Reporting Format,CH4,1.A.3.b.i,1.A.3.b.i. Cars,1.A.3.b,Gg,2022,0.54293,NaN,20260315,EEA,BGR
1,BG,Bulgaria,IPCC Common Reporting Format,CH4,1.A.3.b.i,1.A.3.b.i. Cars,1.A.3.b,Gg,2020,0.54624,NaN,20260315,EEA,BGR
2,BE,Belgium,IPCC Common Reporting Format,CH4,1.A.3.b.i,1.A.3.b.i. Cars,1.A.3.b,Gg,2023,0.54691,NaN,20260315,EEA,BEL
3,BE,Belgium,IPCC Common Reporting Format,CH4,1.A.3.b.i,1.A.3.b.i. Cars,1.A.3.b,Gg,2019,0.55076,NaN,20260315,EEA,BEL
4,BG,Bulgaria,IPCC Common Reporting Format,CH4,1.A.3.b.i,1.A.3.b.i. Cars,1.A.3.b,Gg,2021,0.55687,NaN,20260315,EEA,BGR
5,BE,Belgium,IPCC Common Reporting Format,CH4,1.A.3.b.i,1.A.3.b.i. Cars,1.A.3.b,Gg,2018,0.56147,NaN,20260315,EEA,BEL
6,PT,Portugal,IPCC Common Reporting Format,CH4,1.A.3.b.i,1.A.3.b.i. Cars,1.A.3.b,Gg,2017,0.56205,NaN,20260315,EEA,PRT


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173292 entries, 0 to 173291
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Country_code        173292 non-null  object 
 1   Country             173292 non-null  object 
 2   Format_name         173292 non-null  object 
 3   Pollutant_name      173292 non-null  object 
 4   Sector_code         173292 non-null  object 
 5   Sector_name         173292 non-null  object 
 6   Parent_sector_code  168108 non-null  object 
 7   Unit                173292 non-null  object 
 8   Year                173292 non-null  int64  
 9   emissions           105594 non-null  float64
 10  Notation            61168 non-null   object 
 11  PublicationDate     173292 non-null  int64  
 12  DataSource          173292 non-null  object 
 13  Country_code_3      173292 non-null  object 
dtypes: float64(1), int64(2), object(11)
memory usage: 18.5+ MB


Data Cleaning

In [ ]:
columns_to_drop = [
    'Country_code',
    'Format_name',
    'Parent_sector_code',
    'Unit',
    'Notation',
    'PublicationDate',
    'DataSource'
]
df = df.drop(columns=columns_to_drop)
display(df.head())

,Country,Pollutant_name,Sector_code,Sector_name,Year,emissions,Country_code_3
0,Bulgaria,CH4,1.A.3.b.i,1.A.3.b.i. Cars,2022,0.54293,BGR
1,Bulgaria,CH4,1.A.3.b.i,1.A.3.b.i. Cars,2020,0.54624,BGR
2,Belgium,CH4,1.A.3.b.i,1.A.3.b.i. Cars,2023,0.54691,BEL
3,Belgium,CH4,1.A.3.b.i,1.A.3.b.i. Cars,2019,0.55076,BEL
4,Bulgaria,CH4,1.A.3.b.i,1.A.3.b.i. Cars,2021,0.55687,BGR


In [ ]:
import re

# Function to clean the Sector_name
def clean_sector_name(sector_name):
    if pd.isna(sector_name):
        return sector_name
    # Use regex to find the part after the first set of alphanumeric codes and a dot, potentially followed by spaces
    match = re.search(r'([0-9A-Za-z\.]+\s*)?(.*)', sector_name)
    if match:
        # The second group captures the desired part (e.g., 'Cars', 'Household')
        cleaned_name = match.group(2).strip()
        if cleaned_name.lower().startswith('all greenhouse gases'):
            return cleaned_name # Keep the full name if it's 'All greenhouse gases'
        return cleaned_name
    return sector_name

# Apply the cleaning function to the 'Sector_name' column
df['Sector_name'] = df['Sector_name'].apply(clean_sector_name)

# Display the first few rows to show the cleaned column
display(df[['Sector_name']].head())

,Sector_name
0,Cars
1,Cars
2,Cars
3,Cars
4,Cars


In [ ]:
df = df.drop(columns=['Sector_code'])
display(df)

,Country,Pollutant_name,Sector_name,Year,emissions,Country_code_3
0,Bulgaria,CH4,Cars,2022,0.54293,BGR
1,Bulgaria,CH4,Cars,2020,0.54624,BGR
2,Belgium,CH4,Cars,2023,0.54691,BEL
3,Belgium,CH4,Cars,2019,0.55076,BEL
4,Bulgaria,CH4,Cars,2021,0.55687,BGR
...,...,...,...,...,...,...
173287,Ireland,CH4,Cars,2016,0.50761,IRL
173288,Bulgaria,CH4,Cars,2024,0.51868,BGR
173289,Belgium,CH4,Cars,2022,0.51919,BEL
173290,Bulgaria,CH4,Cars,2023,0.51978,BGR


In [ ]:
# Check for missing values in 'emissions' column
missing_emissions_count = df['emissions'].isnull().sum()
print(f"Number of missing values in 'emissions' column: {missing_emissions_count}")

# Drop rows where 'emissions' is NaN
initial_rows = df.shape[0]
df.dropna(subset=['emissions'], inplace=True)
rows_after_dropping_emissions = df.shape[0]
print(f"Rows dropped due to missing 'emissions': {initial_rows - rows_after_dropping_emissions}")

# Verify no more missing values in 'emissions'
print(f"Missing values in 'emissions' after dropping: {df['emissions'].isnull().sum()}")

Number of missing values in 'emissions' column: 67698
Rows dropped due to missing 'emissions': 67698
Missing values in 'emissions' after dropping: 0


In [ ]:
# Check for duplicate rows
duplicate_rows_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_rows_count}")

# Drop duplicate rows
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
rows_after_dropping_duplicates = df.shape[0]
print(f"Rows dropped due to duplication: {initial_rows - rows_after_dropping_duplicates}")

# Verify no more duplicate rows
print(f"Duplicate rows after dropping: {df.duplicated().sum()}")

Number of duplicate rows found: 0
Rows dropped due to duplication: 0
Duplicate rows after dropping: 0


In [ ]:
# Display DataFrame information to review data types and non-null counts
df.info()

# Display the first few rows of the cleaned DataFrame
display(df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 105594 entries, 0 to 173291
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Country         105594 non-null  object 
 1   Pollutant_name  105594 non-null  object 
 2   Sector_name     105594 non-null  object 
 3   Year            105594 non-null  int64  
 4   emissions       105594 non-null  float64
 5   Country_code_3  105594 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 5.6+ MB


,Country,Pollutant_name,Sector_name,Year,emissions,Country_code_3
0,Bulgaria,CH4,Cars,2022,0.54293,BGR
1,Bulgaria,CH4,Cars,2020,0.54624,BGR
2,Belgium,CH4,Cars,2023,0.54691,BEL
3,Belgium,CH4,Cars,2019,0.55076,BEL
4,Bulgaria,CH4,Cars,2021,0.55687,BGR


In [ ]:
df

,Country,Pollutant_name,Sector_name,Year,emissions,Country_code_3
0,Bulgaria,CH4,Cars,2022,0.54293,BGR
1,Bulgaria,CH4,Cars,2020,0.54624,BGR
2,Belgium,CH4,Cars,2023,0.54691,BEL
3,Belgium,CH4,Cars,2019,0.55076,BEL
4,Bulgaria,CH4,Cars,2021,0.55687,BGR
...,...,...,...,...,...,...
173287,Ireland,CH4,Cars,2016,0.50761,IRL
173288,Bulgaria,CH4,Cars,2024,0.51868,BGR
173289,Belgium,CH4,Cars,2022,0.51919,BEL
173290,Bulgaria,CH4,Cars,2023,0.51978,BGR


In [ ]:
df.tail()

,Country,Pollutant_name,Sector_name,Year,emissions,Country_code_3
173287,Ireland,CH4,Cars,2016,0.50761,IRL
173288,Bulgaria,CH4,Cars,2024,0.51868,BGR
173289,Belgium,CH4,Cars,2022,0.51919,BEL
173290,Bulgaria,CH4,Cars,2023,0.51978,BGR
173291,Portugal,CH4,Cars,2018,0.52237,PRT


### Removing rows with 'all greenhouse gases' in 'Pollutant_name'

I will now filter out all rows where the `Pollutant_name` is 'all greenhouse gases' to focus on specific pollutant data.

In [ ]:
initial_rows = df.shape[0]
df = df[df['Pollutant_name'] != 'All greenhouse gases - (CO2 equivalent)']
rows_after_dropping_ghg = df.shape[0]
print(f"Rows dropped due to 'Pollutant_name' being 'all greenhouse gases': {initial_rows - rows_after_dropping_ghg}")
print(f"Number of rows after dropping: {rows_after_dropping_ghg}")

display(df.head())
df.info()

Rows dropped due to 'Pollutant_name' being 'all greenhouse gases': 33030
Number of rows after dropping: 72564


,Country,Pollutant_name,Sector_name,Year,emissions,Country_code_3
0,Bulgaria,CH4,Cars,2022,0.54293,BGR
1,Bulgaria,CH4,Cars,2020,0.54624,BGR
2,Belgium,CH4,Cars,2023,0.54691,BEL
3,Belgium,CH4,Cars,2019,0.55076,BEL
4,Bulgaria,CH4,Cars,2021,0.55687,BGR


<class 'pandas.core.frame.DataFrame'>
Index: 72564 entries, 0 to 173291
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country         72564 non-null  object 
 1   Pollutant_name  72564 non-null  object 
 2   Sector_name     72564 non-null  object 
 3   Year            72564 non-null  int64  
 4   emissions       72564 non-null  float64
 5   Country_code_3  72564 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 3.9+ MB


In [ ]:
df

,Country,Pollutant_name,Sector_name,Year,emissions,Country_code_3
0,Bulgaria,CH4,Cars,2022,0.54293,BGR
1,Bulgaria,CH4,Cars,2020,0.54624,BGR
2,Belgium,CH4,Cars,2023,0.54691,BEL
3,Belgium,CH4,Cars,2019,0.55076,BEL
4,Bulgaria,CH4,Cars,2021,0.55687,BGR
...,...,...,...,...,...,...
173287,Ireland,CH4,Cars,2016,0.50761,IRL
173288,Bulgaria,CH4,Cars,2024,0.51868,BGR
173289,Belgium,CH4,Cars,2022,0.51919,BEL
173290,Bulgaria,CH4,Cars,2023,0.51978,BGR


In [ ]:
# Save the cleaned DataFrame to a CSV file
df.to_csv('cleaned_data.csv', index=False)
print("DataFrame saved to 'cleaned_data.csv'")

DataFrame saved to 'cleaned_data.csv'


## Visualizing Emissions Data with Plotly

This section will generate five different interactive visualizations using Plotly, based on the cleaned emissions data. We will start by loading the `cleaned_data.csv` file and setting up a consistent Plotly template.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Load the cleaned data
df = pd.read_csv('cleaned_data.csv')

# Set a consistent Plotly template
plotly_template = 'plotly_white'

print("Data loaded and Plotly template set.")
display(df.head())

Data loaded and Plotly template set.


,Country,Pollutant_name,Sector_name,Year,emissions,Country_code_3
0,Bulgaria,CH4,Cars,2022,0.54293,BGR
1,Bulgaria,CH4,Cars,2020,0.54624,BGR
2,Belgium,CH4,Cars,2023,0.54691,BEL
3,Belgium,CH4,Cars,2019,0.55076,BEL
4,Bulgaria,CH4,Cars,2021,0.55687,BGR


### 1. Choropleth Map: Total Emissions per Country (Europe, Selected Year & Pollutant)

This map visualizes the total emissions for a selected year and pollutant across European countries. The intensity of the color represents the emission level, using `Country_code_3` for geographical mapping.

In [ ]:
# Define selection for the map
selected_year_map = 2022
selected_pollutant_map = 'CH4'

# Filter and aggregate data for the choropleth map
df_map = df[(df['Year'] == selected_year_map) & (df['Pollutant_name'] == selected_pollutant_map)]
choropleth_data = df_map.groupby(['Country', 'Country_code_3'])['emissions'].sum().reset_index()

# Create the choropleth map
fig_choropleth = px.choropleth(
    choropleth_data,
    locations='Country_code_3',
    color='emissions',
    hover_name='Country',
    color_continuous_scale=px.colors.sequential.Plasma,
    title=f'Total {selected_pollutant_map} Emissions in Europe ({selected_year_map})',
    template=plotly_template,
    scope='europe' # Scope to Europe
)

fig_choropleth.show()

### 2. Time Series Line Chart: Total Emissions (Multiple Countries, Filtered by Pollutant & Sector)

This chart displays the trend of total emissions from 2016 to 2024 for multiple selected countries, filtered by a specific pollutant and sector. A vertical dashed line indicates the year 2020, marked as 'COVID-19' to highlight potential impacts.

In [ ]:
# Define selections for the time series chart
selected_countries_ts = ['Germany', 'France', 'United Kingdom', 'Italy', 'Spain']
selected_pollutant_ts = 'CO2'
selected_sector_ts = 'Cars'

# Filter and aggregate data for the time series chart
df_ts = df[
    (df['Country'].isin(selected_countries_ts)) &
    (df['Pollutant_name'] == selected_pollutant_ts) &
    (df['Sector_name'] == selected_sector_ts)
]
time_series_data = df_ts.groupby(['Year', 'Country'])['emissions'].sum().reset_index()

# Create the time series line chart
fig_ts = px.line(
    time_series_data,
    x='Year',
    y='emissions',
    color='Country',
    title=f'Total {selected_pollutant_ts} Emissions from {selected_sector_ts} (2016-2024)',
    labels={'emissions': 'Total Emissions'},
    template=plotly_template
)

# Add a vertical dashed line for COVID-19 at 2020
fig_ts.add_vline(
x=2020,
line_width=2,
line_dash="dash",
line_color="red",
annotation_text="COVID-19",
annotation_position="top right"
)

fig_ts.show()

### 3. Stacked Bar Chart: Total Emissions per Country (Stacked by Top 10 Sectors)

This stacked bar chart shows the distribution of emissions by sector for each country, for a selected year and pollutant. To maintain readability, it focuses on the top 10 emitting sectors.

In [ ]:
# Define selections for the stacked bar chart
selected_year_stacked = 2023
selected_pollutant_stacked = 'CO2'

# Filter data for the selected year and pollutant
df_stacked = df[
    (df['Year'] == selected_year_stacked) &
    (df['Pollutant_name'] == selected_pollutant_stacked)
]

# Aggregate emissions by country and sector
stacked_bar_data = df_stacked.groupby(['Country', 'Sector_name'])['emissions'].sum().reset_index()

# Identify top 10 sectors by total emissions for readability
top_10_sectors = stacked_bar_data.groupby('Sector_name')['emissions'].sum().nlargest(10).index
stacked_bar_data_filtered = stacked_bar_data[stacked_bar_data['Sector_name'].isin(top_10_sectors)]

# Create the stacked bar chart
fig_stacked = px.bar(
    stacked_bar_data_filtered,
    x='Country',
    y='emissions',
    color='Sector_name',
    title=f'Total {selected_pollutant_stacked} Emissions by Country and Top 10 Sectors ({selected_year_stacked})',
    labels={'emissions': 'Total Emissions'},
    template=plotly_template
)

fig_stacked.show()

### 4. Top N Emitters Horizontal Bar Chart

This horizontal bar chart highlights the top 10 countries with the highest total emissions for a selected year and pollutant. Countries are sorted in descending order of their emissions, making it easy to identify the largest emitters.

In [ ]:
# Define selections for the top emitters chart
selected_year_top_emitters = 2021
selected_pollutant_top_emitters = 'CO2'

# Filter and aggregate data for the top emitters chart
df_top_emitters = df[
    (df['Year'] == selected_year_top_emitters) &
    (df['Pollutant_name'] == selected_pollutant_top_emitters)
]
top_emitters_data = df_top_emitters.groupby('Country')['emissions'].sum().nlargest(10).reset_index()

# Sort data for the horizontal bar chart
top_emitters_data = top_emitters_data.sort_values(by='emissions', ascending=True)

# Create the horizontal bar chart
fig_top_emitters = px.bar(
    top_emitters_data,
    x='emissions',
    y='Country',
    orientation='h',
    title=f'Top 10 Countries by Total {selected_pollutant_top_emitters} Emissions ({selected_year_top_emitters})',
    labels={'emissions': 'Total Emissions'},
    template=plotly_template
)

fig_top_emitters.show()

### 5. Sector vs Country Heatmap

This heatmap provides a visual representation of emissions intensity, showing the total emissions for the top 15 sectors across different countries, filtered by a selected year and pollutant. The color intensity indicates the level of emissions.

In [ ]:
# Define selections for the heatmap
selected_year_heatmap = 2022
selected_pollutant_heatmap = 'N2O'

# Filter data for the selected year and pollutant
df_heatmap = df[
    (df['Year'] == selected_year_heatmap) &
    (df['Pollutant_name'] == selected_pollutant_heatmap)
]

# Aggregate emissions by sector and country
heatmap_data = df_heatmap.groupby(['Sector_name', 'Country'])['emissions'].sum().reset_index()

# Identify top 15 sectors by total emissions
top_15_sectors_heatmap = heatmap_data.groupby('Sector_name')['emissions'].sum().nlargest(15).index
heatmap_data_filtered = heatmap_data[heatmap_data['Sector_name'].isin(top_15_sectors_heatmap)]

# Pivot the data to create the matrix for the heatmap
heatmap_pivot = heatmap_data_filtered.pivot_table(index='Sector_name', columns='Country', values='emissions', fill_value=0)

# Create the heatmap
fig_heatmap = go.Figure(data=go.Heatmap(
    z=heatmap_pivot.values,
    x=heatmap_pivot.columns,
    y=heatmap_pivot.index,
    colorscale=px.colors.sequential.Plasma
))

fig_heatmap.update_layout(
    title=f'Emissions Heatmap: Top 15 Sectors vs Countries ({selected_pollutant_heatmap}, {selected_year_heatmap})',
    xaxis_title='Country',
    yaxis_title='Sector Name',
    template=plotly_template
)

fig_heatmap.show()

### Streamlit Application (`app.py`)

This code will create a Streamlit application that hosts all five visualizations with interactive filters in the sidebar. You can save this code as `app.py` and run it using `streamlit run app.py`.

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# --- Page Configuration ---
st.set_page_config(layout="wide")

# --- Title of the App ---
st.title("GHG Emissions Data Analysis")

# --- Data Loading ---
@st.cache_data
def load_data():
    df = pd.read_csv('cleaned_data.csv')
    # The cleaned_data.csv should already have 'All greenhouse gases - (CO2 equivalent)' removed
    # as per previous notebook steps, but adding this line for robustness just in case.
    df = df[df['Pollutant_name'] != 'All greenhouse gases - (CO2 equivalent)']
    return df

df = load_data()

# --- Sidebar Filters ---
st.sidebar.header("Filter Options")

# Year Slider
min_year = int(df['Year'].min())
max_year = int(df['Year'].max())
selected_year = st.sidebar.slider("Select Year", min_value=min_year, max_value=max_year, value=min_year)

# Pollutant Dropdown
pollutant_options = sorted(df['Pollutant_name'].unique())
selected_pollutant = st.sidebar.selectbox("Select Pollutant", pollutant_options, index=pollutant_options.index('CH4') if 'CH4' in pollutant_options else 0)

# Sector Dropdown
sector_options = sorted(df['Sector_name'].unique())
selected_sector = st.sidebar.selectbox("Select Sector", sector_options, index=sector_options.index('Cars') if 'Cars' in sector_options else 0)

# Country Multi-select for Time Series
all_countries = sorted(df['Country'].unique())
default_countries_ts = ['Germany', 'France', 'Italy', 'Spain', 'United Kingdom']
# Filter default countries to only include those present in the dataset
default_countries_ts = [c for c in default_countries_ts if c in all_countries]
selected_countries_ts = st.sidebar.multiselect(
    "Select Countries for Time Series",
    all_countries,
    default=default_countries_ts
)

# --- Plotly Template ---
plotly_template = 'plotly_white'

# --- Visualisation 1: Choropleth Map ---
st.subheader("1. Total Emissions per Country (Europe, Selected Year & Pollutant)")
df_map = df[(df['Year'] == selected_year) & (df['Pollutant_name'] == selected_pollutant)]
choropleth_data = df_map.groupby(['Country', 'Country_code_3'])['emissions'].sum().reset_index()

fig_choropleth = px.choropleth(
    choropleth_data,
    locations='Country_code_3',
    color='emissions',
    hover_name='Country',
    color_continuous_scale=px.colors.sequential.Plasma,
    title=f'Total {selected_pollutant} Emissions in Europe ({selected_year})',
    template=plotly_template,
    scope='europe'
)
st.plotly_chart(fig_choropleth, use_container_width=True)

# --- Visualisation 2: Time Series Line Chart ---
st.subheader("2. Total Emissions: Multiple Countries, Filtered by Pollutant & Sector (2016-2024)")
if selected_countries_ts:
    df_ts = df[
        (df['Country'].isin(selected_countries_ts)) &
        (df['Pollutant_name'] == selected_pollutant) &
        (df['Sector_name'] == selected_sector)
    ]
    time_series_data = df_ts.groupby(['Year', 'Country'])['emissions'].sum().reset_index()

    fig_ts = px.line(
        time_series_data,
        x='Year',
        y='emissions',
        color='Country',
        title=f'Total {selected_pollutant} Emissions from {selected_sector} (2016-2024)',
        labels={'emissions': 'Total Emissions'},
        template=plotly_template
    )

    fig_ts.add_vline(
        x=2020,
        line_width=2,
        line_dash="dash",
        line_color="red",
        annotation_text="COVID-19",
        annotation_position="top right"
    )
    st.plotly_chart(fig_ts, use_container_width=True)
else:
    st.info("Please select countries for the Time Series Line Chart from the sidebar.")

# --- Visualisation 3: Stacked Bar Chart ---
st.subheader("3. Total Emissions per Country (Stacked by Top 10 Sectors)")
df_stacked = df[
    (df['Year'] == selected_year) &
    (df['Pollutant_name'] == selected_pollutant)
]

stacked_bar_data = df_stacked.groupby(['Country', 'Sector_name'])['emissions'].sum().reset_index()

# Identify top 10 sectors by total emissions for readability
top_10_sectors = stacked_bar_data.groupby('Sector_name')['emissions'].sum().nlargest(10).index
stacked_bar_data_filtered = stacked_bar_data[stacked_bar_data['Sector_name'].isin(top_10_sectors)]

fig_stacked = px.bar(
    stacked_bar_data_filtered,
    x='Country',
    y='emissions',
    color='Sector_name',
    title=f'Total {selected_pollutant} Emissions by Country and Top 10 Sectors ({selected_year})',
    labels={'emissions': 'Total Emissions'},
    template=plotly_template
)
st.plotly_chart(fig_stacked, use_container_width=True)

# --- Visualisation 4: Top N Emitters Horizontal Bar Chart ---
st.subheader("4. Top 10 Emitters by Total Emissions")
df_top_emitters = df[
    (df['Year'] == selected_year) &
    (df['Pollutant_name'] == selected_pollutant)
]
top_emitters_data = df_top_emitters.groupby('Country')['emissions'].sum().nlargest(10).reset_index()

# Sort data for the horizontal bar chart
top_emitters_data = top_emitters_data.sort_values(by='emissions', ascending=True)

fig_top_emitters = px.bar(
    top_emitters_data,
    x='emissions',
    y='Country',
    orientation='h',
    title=f'Top 10 Countries by Total {selected_pollutant} Emissions ({selected_year})',
    labels={'emissions': 'Total Emissions'},
    template=plotly_template
)
st.plotly_chart(fig_top_emitters, use_container_width=True)

# --- Visualisation 5: Sector vs Country Heatmap ---
st.subheader("5. Emissions Heatmap: Top 15 Sectors vs Countries")
df_heatmap = df[
    (df['Year'] == selected_year) &
    (df['Pollutant_name'] == selected_pollutant)
]

# Aggregate emissions by sector and country
heatmap_data = df_heatmap.groupby(['Sector_name', 'Country'])['emissions'].sum().reset_index()

# Identify top 15 sectors by total emissions
top_15_sectors_heatmap = heatmap_data.groupby('Sector_name')['emissions'].sum().nlargest(15).index
heatmap_data_filtered = heatmap_data[heatmap_data['Sector_name'].isin(top_15_sectors_heatmap)]

# Pivot the data to create the matrix for the heatmap
if not heatmap_data_filtered.empty:
    heatmap_pivot = heatmap_data_filtered.pivot_table(index='Sector_name', columns='Country', values='emissions', fill_value=0)

    fig_heatmap = go.Figure(data=go.Heatmap(
        z=heatmap_pivot.values,
        x=heatmap_pivot.columns,
        y=heatmap_pivot.index,
        colorscale=px.colors.sequential.Plasma
    ))

    fig_heatmap.update_layout(
        title=f'Emissions Heatmap: Top 15 Sectors vs Countries ({selected_pollutant}, {selected_year})',
        xaxis_title='Country',
        yaxis_title='Sector Name',
        template=plotly_template
    )
    st.plotly_chart(fig_heatmap, use_container_width=True)
else:
    st.info(f"No data available for heatmap with selected year ({selected_year}) and pollutant ({selected_pollutant}).")

Writing app.py


### `requirements.txt`

Save this content as `requirements.txt` in the same directory as your `app.py` for Streamlit Cloud deployment.

In [ ]:
%%writefile requirements.txt
pandas
plotly
streamlit

Writing requirements.txt


In [ ]:
!pip install -r requirements.txt
!streamlit run app.py & npx localtunnel --port 8501

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 77.7 MB/s eta 0:00:00


⠙⠹⠸⠼⠴⠦⠧⠇Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.116.185.68:8501

